In [26]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import datasets, transforms
import os
from torch.utils.data import DataLoader, random_split

from sklearn.metrics import classification_report,confusion_matrix


In [27]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

dataset = datasets.ImageFolder('aug_processed_data', transform=transform)

# Train/val split
base_dir = 'Splited_Data'
train_dataset = datasets.ImageFolder(os.path.join(base_dir, 'train'), transform=transform)
val_dataset = datasets.ImageFolder(os.path.join(base_dir, 'val'), transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

num_classes = 2

In [28]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.fc1 = nn.Linear(32 * 32 * 32, 64)
        self.fc2 = nn.Linear(64, 2)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # 64x64
        x = self.pool(F.relu(self.conv2(x)))   # 32x32
        x = x.view(-1, 32 * 32 * 32)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

In [29]:
## Approch 2


class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.pool = nn.MaxPool2d(2, 2)

        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)

        self.dropout = nn.Dropout(p=0.5)  # Dropout with 50% probability

        self.fc1 = nn.Linear(32 * 32 * 32, 64)
        self.fc2 = nn.Linear(64, 2)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))   # 128->64
        x = self.pool(F.relu(self.bn2(self.conv2(x))))   # 64->32

        x = x.view(-1, 32 * 32 * 32)
        x = self.dropout(F.relu(self.fc1(x)))            # Dropout before fc2
        x = self.fc2(x)
        return x

In [30]:
model = SimpleCNN()

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)  # weight_decay added

In [31]:
def evaluate(model, data_loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in data_loader:
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_loss = total_loss / len(data_loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy


def train_model(epochs=10):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        correct = 0
        total = 0

        for images, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            print(f"Batch loss: {loss.item()}")
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_loss = total_loss / len(train_loader)
        train_acc = 100 * correct / total

        val_loss, val_acc = evaluate(model, val_loader, criterion)

        print(f"Epoch {epoch+1} | "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

# Assuming you already have a val_loader for your validation dataset

train_model(epochs=100)

Batch loss: 0.8179885745048523
Batch loss: 12.964102745056152
Batch loss: 2.570478677749634
Batch loss: 2.460704803466797
Batch loss: 2.098999261856079
Epoch 1 | Train Loss: 4.1825 | Train Acc: 49.38% | Val Loss: 0.5714 | Val Acc: 75.00%
Batch loss: 2.351388931274414
Batch loss: 2.2749791145324707
Batch loss: 1.0624151229858398
Batch loss: 1.0426865816116333
Batch loss: 1.030281901359558
Epoch 2 | Train Loss: 1.5524 | Train Acc: 65.00% | Val Loss: 0.4885 | Val Acc: 55.00%
Batch loss: 0.9084410667419434
Batch loss: 1.245341420173645
Batch loss: 0.4884068965911865
Batch loss: 0.5702533721923828
Batch loss: 0.7711533904075623
Epoch 3 | Train Loss: 0.7967 | Train Acc: 68.12% | Val Loss: 0.4230 | Val Acc: 80.00%
Batch loss: 0.43216216564178467
Batch loss: 0.32847821712493896
Batch loss: 0.6459627747535706
Batch loss: 0.3872631788253784
Batch loss: 0.5436936020851135
Epoch 4 | Train Loss: 0.4675 | Train Acc: 78.75% | Val Loss: 0.4037 | Val Acc: 72.50%
Batch loss: 0.3381339907646179
Batch los

Model Testing

In [32]:

def predict_single_image(image_path, model, class_names):
    model.eval()
    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor()
    ])

    img = Image.open(image_path).convert("RGB")
    img_tensor = transform(img).unsqueeze(0)  # Add batch dimension

    with torch.no_grad():
        output = model(img_tensor)
        probs = F.softmax(output, dim=1)
        _, predicted = torch.max(probs, 1)

    print(f"Predicted Class: {class_names[predicted.item()]}")
    print(f"Class Probabilities: {probs.squeeze().numpy()}")


In [33]:
# Assuming dataset = ImageFolder(...)
class_names = dataset.classes  # ['healthy', 'infected']

# Path to one test image
test_image_path_1 = "processed_data/serie healthy leaves/healthy_04.png"

predict_single_image(test_image_path_1, model, class_names)

Predicted Class: serie_healthy_leaves
Class Probabilities: [9.999999e-01 7.000046e-08]


In [34]:
test_image_path_2 = "processed_data/serie infected leaves/infected_05.png"
predict_single_image(test_image_path_2, model, class_names)


Predicted Class: series_infected_leaves
Class Probabilities: [6.0891155e-16 1.0000000e+00]


In [36]:
def evaluate_final_model():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print("\n📊 Final Evaluation on Validation Set:")
    print(classification_report(all_labels, all_preds, target_names=val_dataset.classes, digits=2))

# Run this after training
print("Model Evaluation  of Simple CNN")
evaluate_final_model()

Model Evaluation  of Simple CNN

📊 Final Evaluation on Validation Set:
                        precision    recall  f1-score   support

  serie_healthy_leaves       0.95      1.00      0.98        20
series_infected_leaves       1.00      0.95      0.97        20

              accuracy                           0.97        40
             macro avg       0.98      0.97      0.97        40
          weighted avg       0.98      0.97      0.97        40

